In [ ]:
"""
AMBS-SD (online sequencing) heuristic benchmark for the sequence-dependent
Stochastic Capacitated Lot-Sizing Problem (S-CLSP-SD).

This script provides a baseline heuristic aligned with the DeepRL evaluation
protocol, but explicitly accounting for sequence-dependent setup times (st_sd)
and setup costs (k_sd).

Main components
---------------
- SD instance generation (K=10, 7 benchmark cases) with st_sd and k_sd.
- Discrete-uniform demand generator consistent with the environment.
- EOQ-based AMBS thresholds (using legacy per-product k for calibration).
- Online AMBS-SD heuristic:
    * Step 1 selects the next product based on expected backorder and a
      transition weight w(pred, i), where pred is the current setup state.
    * Real SD setup-time and setup-cost are accumulated along the path
      iSU -> seq[0] -> ... -> seq[-1].
    * Step 3 adds batches only to already selected products (no resequencing).
- Grid search over (xb, xh, zmax, lam_c, lam_t).
- Final evaluation over 10 runs (horizon=10000, warmup=1000).

Capacity accounting:
- Each batch consumes 1 unit.
- Each setup transition u -> v consumes st_sd[u, v] units and incurs k_sd[u, v] cost.
"""


import os
import math
import random
import numpy as np
from tqdm import tqdm

# =======================
# 0) Global configuration
# =======================
SEED = 123
INSTANCE_SEED = 123  # seed for st_sd / k_sd generation 

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

np.random.seed(SEED)
random.seed(SEED)

# ============================================================
# 1) Select WHICH SD instance to run (K=10)
# ============================================================
# 3.1) Two families, mu=4 for all, COV low, cf=1.1
# 3.2) Two families, mu=4 for all, COV high, cf=1.1
# 3.3) Two families, fam A mu=2, fam B mu=8, COV high, cf=1.1
# 3.4) Two critical products, mu=4 for all, COV low, cf=1.1
# 3.5) Two critical products, mu=4 for all, COV high, cf=1.1
# 3.6) Two critical products, crit mu=2 and mu=8, rest mu=4, COV high, cf=1.1
# 3.7) Fully heterogeneous transitions, mu=[2..11], COV high, cf=1.1
INSTANCE_ID = "3.1"
  
# =======================
# 2) Common scales / knobs
# =======================
K = 10
CF = 1.1

COV_LOW = 0.14
COV_HIGH = 0.58

# SD setup scaling (time) and cost limits
SF = 0.20
SETUP_COST_MAX = 200

# Cost / batch parameters
H_HOLD = 1.0
B_BACK = 9.0
BS_VAL = 10

# Legacy TBO for k vector used in EOQ/thresholds (AMBS thresholds)
TBO = 10.0


# ==========================
# 3) SD instance builders 
# ==========================

def compute_CAP(mu, bs_val, cf):
    """
    Compute the per-period capacity in batch units:

        CAP = ceil(cf * (sum(mu) / bs_val))

    :param mu: Mean demand vector.
    :type mu: numpy.ndarray
    :param bs_val: Batch size (scalar, assumed common across products).
    :type bs_val: int or float
    :param cf: Capacity factor.
    :type cf: float
    :return: Period capacity in batch units.
    :rtype: int
    """
    return int(np.ceil(cf * (mu.sum() / bs_val)))


def _st_float(sf, CAP, K):
    """
    Convert a dimensionless setup-time scaling factor into
    capacity units (batches), proportional to CAP / K.

    :param sf: Setup scaling factor.
    :type sf: float
    :param CAP: Total period capacity.
    :type CAP: int
    :param K: Number of products.
    :type K: int
    :return: Setup time expressed in batch-capacity units.
    :rtype: float
    """
    if sf <= 0:
        return 0.0
    return sf * CAP / K


def build_st_sd_matrix(sd_structure_id, K, CAP, sf_global, seed=123):
    """
    Build the sequence-dependent setup-time matrix st_sd ∈ R^{K×K}.

    sd_structure_id:
      1) Two-family structure
      2) Two critical products
      3) Fully heterogeneous

    Setup times are expressed in batch-capacity units and the diagonal is zero.

    :param sd_structure_id: Identifier of the SD structure.
    :type sd_structure_id: int
    :param K: Number of products.
    :type K: int
    :param CAP: Period capacity (batch units).
    :type CAP: int
    :param sf_global: Global setup-time scaling factor.
    :type sf_global: float
    :param seed: RNG seed (used for heterogeneous case).
    :type seed: int
    :return: Sequence-dependent setup-time matrix of shape (K, K).
    :rtype: numpy.ndarray
    """
    assert sd_structure_id in (1, 2, 3)
    st = np.zeros((K, K), dtype=np.float32)

    if sd_structure_id == 1:
        fam1 = set(range(0, K // 2))
        fam2 = set(range(K // 2, K))

        s = sf_global / 0.20
        sf11 = 0.30 * s
        sf22 = 0.30 * s
        sf12 = 0.80 * s

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                if (i in fam1) and (j in fam1):
                    sf_ij = sf11
                elif (i in fam2) and (j in fam2):
                    sf_ij = sf22
                else:
                    sf_ij = sf12
                st[i, j] = round(_st_float(sf_ij, CAP, K), 1)

        np.fill_diagonal(st, 0.0)
        return st

    if sd_structure_id == 2:
        crit1, crit2 = 1, 7

        s = sf_global / 0.20
        sf_base  = 0.35 * s
        sf_to_c1 = 0.95 * s
        sf_to_c2 = 0.75 * s
        sf_c1c2  = 1.00 * s

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                if (i == crit1 and j == crit2) or (i == crit2 and j == crit1):
                    sf_ij = sf_c1c2
                elif (i == crit1) or (j == crit1):
                    sf_ij = sf_to_c1
                elif (i == crit2) or (j == crit2):
                    sf_ij = sf_to_c2
                else:
                    sf_ij = sf_base
                st[i, j] = round(_st_float(sf_ij, CAP, K), 1)

        np.fill_diagonal(st, 0.0)
        return st

    # sd_structure_id == 3
    rng = np.random.default_rng(seed)
    sf_mat = rng.uniform(0.15, 0.60, size=(K, K)).astype(np.float32)
    np.fill_diagonal(sf_mat, 0.0)

    s = sf_global / 0.20
    for i in range(K):
        for j in range(K):
            st[i, j] = round(_st_float(sf_mat[i, j] * s, CAP, K), 1)

    np.fill_diagonal(st, 0.0)
    return st

def _clip_int(x, lo, hi):
    """
    Round a value to the nearest integer and clip it to a closed interval [lo, hi].

    :param x: Input value to be rounded and clipped.
    :type x: float or int
    :param lo: Lower bound (inclusive).
    :type lo: int
    :param hi: Upper bound (inclusive).
    :type hi: int
    :return: Rounded and clipped integer.
    :rtype: int
    """
    return int(np.clip(np.rint(x), lo, hi))


def build_k_sd_matrix(sd_structure_id, K, setup_cost_max=200, seed=123):
    """
    Build the sequence-dependent setup-cost matrix k_sd ∈ Z^{K×K}.

    sd_structure_id:
      1) Two-family structure (low within-family, high cross-family)
      2) Two critical products (highest costs between the two critical items)
      3) Fully heterogeneous (random costs per transition)

    The diagonal is zero and all entries are clipped to [0, setup_cost_max].

    :param sd_structure_id: Identifier of the SD structure.
    :type sd_structure_id: int
    :param K: Number of products.
    :type K: int
    :param setup_cost_max: Maximum setup cost allowed for any transition.
    :type setup_cost_max: int
    :param seed: RNG seed controlling the random draws.
    :type seed: int
    :return: Sequence-dependent setup-cost matrix of shape (K, K).
    :rtype: numpy.ndarray
    """
    assert sd_structure_id in (1, 2, 3)
    rng = np.random.default_rng(seed)
    ksd = np.zeros((K, K), dtype=np.int32)

    if sd_structure_id == 1:
        fam1 = set(range(0, K // 2))
        fam2 = set(range(K // 2, K))

        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                if (i in fam1) and (j in fam1):
                    base = 60.0 + rng.normal(0, 4.0)
                elif (i in fam2) and (j in fam2):
                    base = 140.0 + rng.normal(0, 6.0)
                else:
                    base = 190.0 + rng.normal(0, 5.0)
                ksd[i, j] = _clip_int(base, 0, setup_cost_max)

        np.fill_diagonal(ksd, 0)
        return ksd

    if sd_structure_id == 2:
        crit1, crit2 = 1, 7
        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                if (i == crit1 and j == crit2) or (i == crit2 and j == crit1):
                    base = setup_cost_max + rng.normal(0, 2.0)
                elif (i == crit1) or (j == crit1):
                    base = 190.0 + rng.normal(0, 5.0)
                elif (i == crit2) or (j == crit2):
                    base = 150.0 + rng.normal(0, 7.0)
                else:
                    base = 80.0 + rng.normal(0, 6.0)
                ksd[i, j] = _clip_int(base, 0, setup_cost_max)

        np.fill_diagonal(ksd, 0)
        return ksd

    # sd_structure_id == 3
    raw = rng.uniform(20.0, setup_cost_max, size=(K, K))
    np.fill_diagonal(raw, 0.0)
    for i in range(K):
        for j in range(K):
            ksd[i, j] = _clip_int(raw[i, j], 0, setup_cost_max)

    np.fill_diagonal(ksd, 0)
    return ksd


def build_mu_eval_case(eval_case_id, K=10):
    """
    Return the mean-demand vector μ for one of the 7 SD evaluation cases (K=10).

    Cases:
      1,2,4,5) homogeneous μ=4
      3) two families (μ=2 for first 5, μ=8 for last 5)
      6) two critical products with μ=[...,2,...,8,...] and the rest μ=4
      7) strictly increasing μ=[2,3,...,11]

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int
    :param K: Number of products (expected to be 10 in this benchmark suite).
    :type K: int
    :return: Mean-demand vector of shape (K,).
    :rtype: numpy.ndarray
    """
    assert K == 10

    if eval_case_id in (1, 2, 4, 5):
        return np.full(K, 4.0, dtype=np.float32)

    if eval_case_id == 3:
        return np.array([2.0] * 5 + [8.0] * 5, dtype=np.float32)

    if eval_case_id == 6:
        mu = np.full(K, 4.0, dtype=np.float32)
        crit1, crit2 = 1, 7
        mu[crit1] = 2.0
        mu[crit2] = 8.0
        return mu

    if eval_case_id == 7:
        return np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype=np.float32)

    raise ValueError("eval_case_id must be in {1..7}")


def build_cov_eval_case(eval_case_id, K=10):
    """
    Return the demand coefficient-of-variation vector (cov) for a given SD eval case.

    By design:
      - low variability only for cases 1 and 4,
      - high variability for the rest.

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int
    :param K: Number of products (expected to be 10 in this benchmark suite).
    :type K: int
    :return: cov vector of shape (K,).
    :rtype: numpy.ndarray
    """
    cov_val = COV_LOW if eval_case_id in (1, 4) else COV_HIGH
    return np.full(K, cov_val, dtype=np.float32)


def sd_structure_id_from_eval_case(eval_case_id):
    """
    Map each evaluation case to the SD structure family used to generate st_sd and k_sd.

    Mapping:
      - cases 1,2,3 -> two-family structure (id=1)
      - cases 4,5,6 -> two-critical-products structure (id=2)
      - case 7      -> fully heterogeneous structure (id=3)

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int
    :return: SD structure identifier in {1,2,3}.
    :rtype: int
    """
    if eval_case_id in (1, 2, 3):
        return 1
    if eval_case_id in (4, 5, 6):
        return 2
    if eval_case_id == 7:
        return 3
    raise ValueError("eval_case_id must be in {1..7}")

# ===============================
# 4) Instance container (SD)
# ===============================
class CLSPInstanceSD:
    """
    Lightweight container for an SD-CLSP instance used by the AMBS-SD benchmark.

    This class mirrors the parameterization of the SD environment:
      - Demand/cost vectors (mu, cov, h, b, bs),
      - Sequence-dependent setup structures (st_sd, k_sd),
      - A legacy per-product setup-cost vector k (used only to compute EOQ/thresholds).

    Capacity is expressed in "batches-equivalent units" and follows the same convention
    used across the codebase:

        CAP = ceil(capacity_factor * (sum(mu) / bs[0]))

    :param K: Number of products.
    :type K: int
    :param capacity_factor: Capacity multiplier used to derive per-period capacity.
    :type capacity_factor: float
    :param mu: Mean-demand vector (shape (K,)).
    :type mu: array-like
    :param bs: Batch-size vector in units per batch (shape (K,)).
    :type bs: array-like
    :param h: Holding-cost vector (shape (K,)).
    :type h: array-like
    :param b: Backorder-cost vector (shape (K,)).
    :type b: array-like
    :param cov: Demand coefficient-of-variation proxy vector (shape (K,)).
    :type cov: array-like
    :param st_sd: Sequence-dependent setup-time matrix (shape (K,K), diag=0).
    :type st_sd: array-like
    :param k_sd: Sequence-dependent setup-cost matrix (shape (K,K), diag=0).
    :type k_sd: array-like
    :param k_legacy: Legacy per-product setup-cost vector (shape (K,)) for EOQ/thresholds.
    :type k_legacy: array-like
    """
    def __init__(self, K, capacity_factor, mu, bs, h, b, cov, st_sd, k_sd, k_legacy):
        self.K = int(K)
        self.capacity_factor = float(capacity_factor)
        self.mu = np.asarray(mu, dtype=np.float32)
        self.bs = np.asarray(bs, dtype=np.int32)
        self.h  = np.asarray(h, dtype=np.float32)
        self.b  = np.asarray(b, dtype=np.float32)
        self.cov = np.asarray(cov, dtype=np.float32)

        # SD transition structures (diag should be 0; enforced by builders)
        self.st_sd = np.asarray(st_sd, dtype=np.float32)  # (K,K) float
        self.k_sd  = np.asarray(k_sd, dtype=np.int32)     # (K,K) int

        # Legacy per-item setup-cost vector used for EOQ/threshold calibration
        self.k = np.asarray(k_legacy, dtype=np.float32)

        assert self.mu.shape == (self.K,)
        assert self.bs.shape == (self.K,)
        assert self.h.shape  == (self.K,)
        assert self.b.shape  == (self.K,)
        assert self.cov.shape == (self.K,)
        assert self.k.shape == (self.K,)
        assert self.st_sd.shape == (self.K, self.K)
        assert self.k_sd.shape  == (self.K, self.K)

    def capacity(self):
        """
        Compute per-period capacity CAP in batches-equivalent units.

        :return: Per-period capacity.
        :rtype: int
        """
        cap = math.ceil(self.capacity_factor * (float(self.mu.sum()) / float(self.bs[0])))
        return int(cap)


def build_inst_sd_eval_case(eval_case_id):
    """
    Build a CLSPInstanceSD for one of the 7 SD evaluation cases (K=10).

    This assembles:
      - Case-dependent demand parameters (mu, cov),
      - Derived capacity CAP (used to scale st_sd),
      - SD setup matrices (st_sd, k_sd),
      - Legacy setup-cost vector k_legacy (EOQ/thresholds only).

    :param eval_case_id: Evaluation case identifier in {1, ..., 7}.
    :type eval_case_id: int
    :return: Fully specified SD instance container.
    :rtype: CLSPInstanceSD
    """
    mu = build_mu_eval_case(eval_case_id, K=K)
    cov = build_cov_eval_case(eval_case_id, K=K)

    CAP = compute_CAP(mu, BS_VAL, CF)
    sd_id = sd_structure_id_from_eval_case(eval_case_id)

    st_sd = build_st_sd_matrix(sd_id, K, CAP, SF, seed=INSTANCE_SEED)
    k_sd  = build_k_sd_matrix(sd_id, K, setup_cost_max=SETUP_COST_MAX, seed=INSTANCE_SEED)

    bs = np.full(K, BS_VAL, dtype=np.int32)
    h  = np.full(K, H_HOLD, dtype=np.float32)
    b  = np.full(K, B_BACK, dtype=np.float32)

    # Legacy per-product setup cost (fixed TBO) used for EOQ/thresholds
    k_legacy = H_HOLD * (TBO ** 2) * mu / 2.0

    inst = CLSPInstanceSD(
        K=K, capacity_factor=CF,
        mu=mu, bs=bs, h=h, b=b, cov=cov,
        st_sd=st_sd, k_sd=k_sd, k_legacy=k_legacy
    )
    return inst


CASE_III_SD_REGISTRY = {
    "3.1": dict(eval_case_id=1),
    "3.2": dict(eval_case_id=2),
    "3.3": dict(eval_case_id=3),
    "3.4": dict(eval_case_id=4),
    "3.5": dict(eval_case_id=5),
    "3.6": dict(eval_case_id=6),
    "3.7": dict(eval_case_id=7),
}

def build_inst_from_instance_id(instance_id):
    instance_id = instance_id.strip()
    if instance_id in CASE_III_SD_REGISTRY:
        p = CASE_III_SD_REGISTRY[instance_id]
        inst = build_inst_sd_eval_case(p["eval_case_id"])
        exp_name = f"ID{instance_id}_EVALcase{p['eval_case_id']}_K{inst.K}_sf{SF}_cf{inst.capacity_factor}"
        return inst, exp_name
    raise ValueError(f"Unknown INSTANCE_ID '{instance_id}' for S-CLSP-SD.")


# ======================
# 5) Demand generator 
# ======================
class DemandGenerator:
    """
    Discrete-uniform demand generator aligned with the SD environment.

    For each product i, demands are sampled from an integer uniform distribution
    d_i ~ U{a_i, ..., b_i}, where the interval [a_i, b_i] is derived from (mu_i, cov_i)
    using the same convention as the env's demand setup.

    Rules:
      - If cov_i ≈ 0.58 (high variability): a_i = 0, b_i = round(2 * mu_i)
      - If cov_i ≈ 0.14 (low variability):
            * if mu_i ≈ 4: a_i = 3, b_i = 5
            * otherwise:   a_i = b_i = round(mu_i)  (degenerate / deterministic)

    :param inst: SD instance containing mu and cov vectors.
    :type inst: CLSPInstanceSD
    :param rng: NumPy RNG used for reproducible sampling.
    :type rng: numpy.random.Generator
    """
    def __init__(self, inst, rng):
        self.inst = inst
        self.rng = rng
        self._setup_params()

    def _setup_params(self):
        """
        Precompute per-product integer demand ranges [a_i, b_i].

        :return: None.
        :rtype: None
        """
        K = self.inst.K
        a = np.empty(K, dtype=np.int32)
        b = np.empty(K, dtype=np.int32)

        for i in range(K):
            cov_i = float(self.inst.cov[i])
            mu_i  = float(self.inst.mu[i])

            if abs(cov_i - 0.58) <= 1e-3:
                a[i] = 0
                b[i] = int(round(2.0 * mu_i))
            else:
                if abs(mu_i - 4.0) <= 1e-3:
                    a[i], b[i] = 3, 5
                else:
                    a[i] = b[i] = int(round(mu_i))

        self._a = a
        self._b = b

    def sample_period(self):
        """
        Sample one demand vector for the current period.

        :return: Integer demand vector (shape (K,)).
        :rtype: numpy.ndarray
        """
        return self.rng.integers(
            low=self._a,
            high=self._b + 1,
            size=self.inst.K,
            dtype=np.int32
        )


# ============================================================
# 6) EOQ / thresholds (same as your AMBS baseline)
# ============================================================
def eoq(mu, k, h):
    """
    Compute the (continuous) Economic Order Quantity (EOQ) per product:

        Q_i = sqrt(2 * mu_i * k_i / h_i)

    :param mu: Mean demand rate(s) (scalar or array).
    :type mu: float or numpy.ndarray
    :param k: (Legacy) setup/fixed cost(s) used for EOQ (scalar or array).
    :type k: float or numpy.ndarray
    :param h: Holding cost rate(s) (scalar or array).
    :type h: float or numpy.ndarray
    :return: EOQ quantities (broadcasted shape).
    :rtype: numpy.ndarray
    """
    return np.sqrt(2.0 * mu * k / h)


def ceoq(mu, k, h):
    """
    EOQ-based average cost proxy used to scale AMBS thresholds:

        C_i = k_i * mu_i / Q_i + h_i * Q_i / 2

    :param mu: Mean demand rate(s).
    :type mu: float or numpy.ndarray
    :param k: (Legacy) setup/fixed cost(s).
    :type k: float or numpy.ndarray
    :param h: Holding cost rate(s).
    :type h: float or numpy.ndarray
    :return: Cost proxy values (broadcasted shape).
    :rtype: numpy.ndarray
    """
    Q = eoq(mu, k, h)
    return k * mu / Q + h * Q / 2.0


def make_thresholds(inst, xb, xh, zmax):
    """
    Build AMBS thresholds (Bmin, Hmax, Zmax) from scaling parameters.

    Notes:
      - Uses inst.k (legacy per-product setup costs) for EOQ/threshold scaling.
      - Zmax limits the number of *new* setups allowed per period in the heuristic.

    :param inst: SD instance providing mu, h and legacy k.
    :type inst: CLSPInstanceSD
    :param xb: Scaling factor for the backorder trigger threshold Bmin.
    :type xb: float
    :param xh: Scaling factor for the holding-cost threshold Hmax.
    :type xh: float
    :param zmax: Maximum number of new setups allowed per period.
    :type zmax: int or float
    :return: Tuple (Bmin, Hmax, Zmax).
    :rtype: tuple[float, float, int]
    """
    C = ceoq(inst.mu, inst.k, inst.h)
    Q = eoq(inst.mu, inst.k, inst.h)
    Bmin = xb * float(np.mean(C))
    Hmax = xh * float(np.sum(inst.h * Q))
    Zmax = int(zmax)
    return (Bmin, Hmax, Zmax)


class AMBSParams:
    """
    Container for AMBS threshold parameters.

    :param Bmin: Backorder trigger threshold.
    :type Bmin: float
    :param Hmax: Holding-cost threshold.
    :type Hmax: float
    :param Zmax: Maximum number of new setups allowed per period.
    :type Zmax: int
    """
    def __init__(self, Bmin, Hmax, Zmax):
        self.Bmin = float(Bmin)
        self.Hmax = float(Hmax)
        self.Zmax = int(Zmax)


class AMBSResult:
    """
    Result summary for one AMBS-SD evaluation rollout.

    :param avg_cost: Mean total cost per period (inventory + backorder + setup).
    :type avg_cost: float
    :param avg_inv_cost: Mean holding cost per period.
    :type avg_inv_cost: float
    :param avg_bo_cost: Mean backorder cost per period.
    :type avg_bo_cost: float
    :param avg_setup_cost: Mean setup cost per period (SD costs).
    :type avg_setup_cost: float
    :param avg_setups: Mean number of setups per period.
    :type avg_setups: float
    :param avg_lot_size_units: Mean produced units per period.
    :type avg_lot_size_units: float
    :param step3_count: Number of periods in which the heuristic executed Step 3.
    :type step3_count: int
    """
    def __init__(self, avg_cost, avg_inv_cost, avg_bo_cost, avg_setup_cost,
                 avg_setups, avg_lot_size_units, step3_count):
        self.avg_cost = float(avg_cost)
        self.avg_inv_cost = float(avg_inv_cost)
        self.avg_bo_cost = float(avg_bo_cost)
        self.avg_setup_cost = float(avg_setup_cost)
        self.avg_setups = float(avg_setups)
        self.avg_lot_size_units = float(avg_lot_size_units)
        self.step3_count = int(step3_count)


# ===============================
# 7) AMBS-SD ONLINE Heuristic
# ===============================
class AMBSParamsSD(AMBSParams):
    """
    Extension of :class:`AMBSParams` with SD transition-aware ranking weights.

    In AMBS-SD Step 1, the next *new* product is selected by:

        score(i) = EBO_cost(i) * w(pred, i)

    where ``pred`` is the current predecessor in the sequence (initially the period
    setup item iSU, then updated each time a new item is appended to the sequence).

    The weight penalizes costly/slow transitions using normalized SD setup cost/time:

      - "inv": w = 1 / (1 + lam_c * k_norm + lam_t * st_norm)
      - "exp": w = exp(-(lam_c * k_norm + lam_t * st_norm))

    :param Bmin: Backorder trigger threshold.
    :type Bmin: float
    :param Hmax: Holding-cost threshold.
    :type Hmax: float
    :param Zmax: Maximum number of new setups allowed per period.
    :type Zmax: int
    :param lam_c: Penalty weight for SD setup costs (dimensionless after normalization).
    :type lam_c: float
    :param lam_t: Penalty weight for SD setup times (dimensionless after normalization).
    :type lam_t: float
    :param weight_mode: Weight function family ("inv" or "exp").
    :type weight_mode: str
    """
    def __init__(self, Bmin, Hmax, Zmax, lam_c=0.0, lam_t=0.0, weight_mode="inv"):
        super().__init__(Bmin, Hmax, Zmax)
        self.lam_c = float(lam_c)
        self.lam_t = float(lam_t)
        assert weight_mode in ("inv", "exp")
        self.weight_mode = weight_mode


class AMBSSDHeuristicOnline:
    """
    SD-aware AMBS heuristic with *online* sequence construction.

    This variant extends the baseline AMBS logic by explicitly tracking the order
    in which new products are introduced within a period:

      - ``iSU``: setup state at the start of the period.
      - ``seq``: ordered list of NEW products appended during Step 1 decisions.
      - ``pred``: predecessor reference used when scoring the next NEW product
                 (starts as iSU, then becomes the last appended item).

    SD setup-time and setup-cost are accounted using the realized transition path:
        iSU -> seq[0] -> ... -> seq[-1]

    :param inst: SD instance with sequence-dependent matrices (st_sd, k_sd).
    :type inst: CLSPInstanceSD
    :param seed: RNG seed controlling initial setup and demand stream.
    :type seed: int
    """
    def __init__(self, inst: CLSPInstanceSD, seed=123):
        self.inst = inst
        self.rng = np.random.default_rng(seed)
        self.dem = DemandGenerator(inst, self.rng)
        self.CAP = inst.capacity()

        # Legacy EOQ uses per-product k (not k_sd) for AMBS-style proxies/thresholds
        self.Q_eoq = eoq(inst.mu, inst.k, inst.h)

        # Normalizers for SD penalties so (lam_c, lam_t) are dimensionless/stable
        mask = ~np.eye(inst.K, dtype=bool)
        ksd = inst.k_sd.astype(np.float32)
        stsd = inst.st_sd.astype(np.float32)

        self.k_norm = float(np.mean(ksd[mask])) if np.any(mask) else 1.0
        self.t_norm = float(np.mean(stsd[mask])) if np.any(mask) else 1.0
        if self.k_norm <= 1e-9:
            self.k_norm = 1.0
        if self.t_norm <= 1e-9:
            self.t_norm = 1.0

    # ----- expected backorder cost under discrete-uniform demand -----
    def _expected_backorder_cost(self, I):
        """
        Compute expected backorder cost per product under discrete-uniform demand:

            E[ b_i * (D - s)^+ ],  D ~ U{a_i, ..., b_i},  s = inventory position.

        Uses the correct support size divisor (b_i - a_i + 1).

        :param I: Inventory vector (positive = on-hand, negative = backlog).
        :type I: numpy.ndarray
        :return: Expected backorder costs per product (shape (K,)).
        :rtype: numpy.ndarray
        """
        K = self.inst.K
        out = np.zeros(K, dtype=np.float64)
        a = self.dem._a
        b = self.dem._b

        for i in range(K):
            s = float(I[i])
            ai = int(a[i])
            bi = int(b[i])

            n_support = (bi - ai + 1)
            if n_support <= 0:
                d = ai
                ebo_units = max(0.0, d - s)
                out[i] = float(self.inst.b[i]) * ebo_units
                continue

            if s >= bi:
                out[i] = 0.0
                continue

            d0 = max(ai, int(math.floor(s)) + 1)
            if d0 > bi:
                out[i] = 0.0
                continue

            n = (bi - d0 + 1)
            sum_d = n * (d0 + bi) / 2.0
            ebo_units = (sum_d - n * s) / float(n_support)

            out[i] = float(self.inst.b[i]) * float(ebo_units)

        return out

    def _il_eoq(self, I):
        """
        IL/EOQ metric used by AMBS tie-breaking / Step 3 logic.

        :param I: Inventory vector.
        :type I: numpy.ndarray
        :return: IL/EOQ values (shape (K,)).
        :rtype: numpy.ndarray
        """
        return I / self.Q_eoq

    # ----- SD sequence accounting -----
    def _sequence_setup_time(self, iSU, seq):
        """
        Compute total SD setup time consumed by the realized sequence.

        :param iSU: Setup item at the start of the period.
        :type iSU: int
        :param seq: Ordered list of appended products (may be empty).
        :type seq: list[int]
        :return: Total setup time (capacity units).
        :rtype: float
        """
        if len(seq) == 0:
            return 0.0
        t = 0.0
        first = seq[0]
        if first != iSU:
            t += float(self.inst.st_sd[iSU, first])
        for u, v in zip(seq[:-1], seq[1:]):
            if u != v:
                t += float(self.inst.st_sd[u, v])
        return t

    def _sequence_setup_cost(self, iSU, seq):
        """
        Compute total SD setup cost incurred by the realized sequence.

        :param iSU: Setup item at the start of the period.
        :type iSU: int
        :param seq: Ordered list of appended products (may be empty).
        :type seq: list[int]
        :return: Total setup cost.
        :rtype: float
        """
        if len(seq) == 0:
            return 0.0
        c = 0.0
        first = seq[0]
        if first != iSU:
            c += float(self.inst.k_sd[iSU, first])
        for u, v in zip(seq[:-1], seq[1:]):
            if u != v:
                c += float(self.inst.k_sd[u, v])
        return c

    def _cap_remaining(self, q, iSU, seq):
        """
        Remaining capacity after batch load and SD setup-time load.

        :param q: Batch quantities produced so far in the period (shape (K,)).
        :type q: numpy.ndarray
        :param iSU: Setup item at the start of the period.
        :type iSU: int
        :param seq: Current realized sequence of appended products.
        :type seq: list[int]
        :return: Remaining capacity (batches-equivalent units).
        :rtype: float
        """
        batch_load = float(np.sum(q))
        setup_load = float(self._sequence_setup_time(iSU, seq))
        return float(self.CAP) - setup_load - batch_load

    def _weight(self, pred, i, params: AMBSParamsSD):
        """
        Transition weight w(pred, i) used in Step 1 scoring.

        :param pred: Predecessor item index.
        :type pred: int
        :param i: Candidate next item index.
        :type i: int
        :param params: SD parameters (lam_c, lam_t, weight_mode).
        :type params: AMBSParamsSD
        :return: Weight in (0, 1] (for positive penalties).
        :rtype: float
        """
        k_ni = float(self.inst.k_sd[pred, i]) / self.k_norm
        t_ni = float(self.inst.st_sd[pred, i]) / self.t_norm
        x = params.lam_c * k_ni + params.lam_t * t_ni
        if params.weight_mode == "exp":
            return float(math.exp(-x))
        return float(1.0 / (1.0 + x))

    def _initial_state_like_deeprl(self):
        """
        Initialize the heuristic state to match the DeepRL env reset.

        The SD env starts each rollout with:
          - zero inventory/backlog (I = 0),
          - a random initial setup item iSU sampled from {0, ..., K-1}.

        :return: Tuple (I0, iSU) with initial inventory and initial setup index.
        :rtype: tuple[numpy.ndarray, int]
        """
        I0 = np.zeros(self.inst.K, dtype=np.int32)
        iSU = int(self.rng.integers(0, self.inst.K))
        return I0, iSU

    def simulate(self, params: AMBSParamsSD, horizon=1000, warmup=100,
                 seed_rollout=None, show_pbar=False):
        """
        Run one AMBS-SD rollout and report average costs after a warmup prefix.

        This simulation mirrors the DeepRL evaluation protocol:
          - fixed horizon with an initial warmup window excluded from averages,
          - discrete-uniform demand sampling via :class:`DemandGenerator`,
          - per-period capacity in batches-equivalent units, where SD setup-times
            consume st_sd[u,v] capacity along the realized sequence.

        Within each period, production is constructed iteratively:
          - Step 1: select product by EBO_cost(i) weighted by SD transition penalty
                    when i is a *new* product (not yet in the sequence).
          - Step 2: if nothing produced yet, optionally produce the current setup item iSU.
          - Step 3: add batches only to already selected products (iSU or in seq).

        :param params: AMBS-SD parameters (thresholds + SD weighting).
        :type params: AMBSParamsSD
        :param horizon: Number of periods to simulate.
        :type horizon: int
        :param warmup: Number of initial periods excluded from averaging.
        :type warmup: int
        :param seed_rollout: Optional rollout seed (typically seed_base + r).
        :type seed_rollout: int or None
        :param show_pbar: If True, show a tqdm progress bar.
        :type show_pbar: bool

        :return: Cost summary averaged over periods t >= warmup.
        :rtype: AMBSResult
        """
        if seed_rollout is not None:
            self.rng = np.random.default_rng(int(seed_rollout))
            self.dem = DemandGenerator(self.inst, self.rng)

        K = self.inst.K
        bs = self.inst.bs
        h = self.inst.h
        bcost = self.inst.b

        I, iSU = self._initial_state_like_deeprl()

        total_cost = inv_cost = bo_cost = setup_cost = 0.0
        total_setups = 0
        total_units = 0
        step3_flags = np.zeros(horizon, dtype=np.int8)

        it = tqdm(range(horizon), desc="AMBS-SD", leave=False) if show_pbar else range(horizon)

        for t in it:
            q = np.zeros(K, dtype=np.int32)

            # Online sequence of NEW products appended in this period (order matters)
            seq = []
            in_seq = set()
            pred = int(iSU)

            produced_any = False

            while True:
                CAP_rem = self._cap_remaining(q, iSU, seq)
                if CAP_rem < 1.0:
                    break

                EBO = self._expected_backorder_cost(I)

                # Priority: plain EBO for already-chosen items; SD-weighted for new candidates
                priority = np.empty(K, dtype=np.float64)
                for i in range(K):
                    if (i == iSU) or (i in in_seq):
                        priority[i] = EBO[i]
                    else:
                        priority[i] = EBO[i] * self._weight(pred, i, params)

                i_best = int(np.argmax(priority))

                # ---------------- STEP 1 ----------------
                if EBO[i_best] > params.Bmin:

                    # Already chosen (or current setup item) -> just add a batch
                    if (i_best == iSU) or (i_best in in_seq):
                        q[i_best] += 1
                        I[i_best] += int(bs[i_best])
                        produced_any = True
                        continue

                    # New product -> append to seq if (i) within Zmax and (ii) setup feasible
                    if len(seq) >= params.Zmax:
                        break

                    st_inc = float(self.inst.st_sd[pred, i_best])
                    CAP_now = self._cap_remaining(q, iSU, seq)
                    if CAP_now < (st_inc + 1.0):
                        break

                    seq.append(i_best)
                    in_seq.add(i_best)
                    pred = int(i_best)

                    q[i_best] += 1
                    I[i_best] += int(bs[i_best])
                    produced_any = True
                    continue

                # ---------------- STEP 2 ----------------
                if not produced_any:
                    hold_cost_proxy = float(bs[iSU] * h[iSU] + np.sum(h * np.maximum(I, 0)))
                    if hold_cost_proxy <= params.Hmax and self._cap_remaining(q, iSU, seq) >= 1.0:
                        q[iSU] += 1
                        I[iSU] += int(bs[iSU])
                        produced_any = True
                        continue
                    else:
                        break

                # ---------------- STEP 3 ----------------
                step3_flags[t] = 1
                cand = [iSU] + list(seq)

                ILEOQ = self._il_eoq(I)
                i_add = int(min(cand, key=lambda i: ILEOQ[i]))

                hold_cost_proxy = float(bs[i_add] * h[i_add] + np.sum(h * np.maximum(I, 0)))
                if hold_cost_proxy <= params.Hmax and self._cap_remaining(q, iSU, seq) >= 1.0:
                    q[i_add] += 1
                    I[i_add] += int(bs[i_add])
                    produced_any = True
                    continue
                else:
                    break

            # --- end fill loop ---

            # SD setup cost induced by iSU -> seq[0] -> ... -> seq[-1]
            setup_c = float(self._sequence_setup_cost(iSU, seq))

            # Count realized setup transitions in the sequence
            if len(seq) == 0:
                n_setups = 0
            else:
                n_setups = 0
                if seq[0] != iSU:
                    n_setups += 1
                n_setups += max(0, len(seq) - 1)

            # Next period setup state = last produced new item (if any)
            iSU_next = int(seq[-1]) if len(seq) > 0 else int(iSU)

            # Demand realization (inventory updates after production decisions)
            I = I - self.dem.sample_period()
            iSU = iSU_next

            # Per-period costs (computed after demand is applied)
            c_inv = float(np.sum(h * np.maximum(I, 0)))
            c_back = float(np.sum(bcost * np.maximum(-I, 0)))
            c_tot = c_inv + c_back + setup_c

            if t >= warmup:
                total_cost += c_tot
                inv_cost += c_inv
                bo_cost += c_back
                setup_cost += setup_c
                total_setups += int(n_setups)
                total_units += int(np.sum(bs * q))

        denom = max(1, int(horizon - warmup))
        return AMBSResult(
            avg_cost=total_cost / denom,
            avg_inv_cost=inv_cost / denom,
            avg_bo_cost=bo_cost / denom,
            avg_setup_cost=setup_cost / denom,
            avg_setups=total_setups / denom,
            avg_lot_size_units=total_units / denom,
            step3_count=int(np.sum(step3_flags)),
        )

    def grid_search(self,
                    xb_grid=None, xh_grid=None, z_grid=None,
                    lam_c_grid=None, lam_t_grid=None,
                    weight_mode="inv",
                    trials=10, horizon=1000, warmup=100, base_seed=12345):
        """
        Grid-search AMBS-SD parameters and return the best configuration by mean cost.

        Each candidate tuple (xb, xh, zmax, lam_c, lam_t) is evaluated over ``trials``
        independent rollouts using seeds ``base_seed + r``. The best configuration is
        then re-evaluated once with a held-out seed (base_seed + 999).

        :param xb_grid: Candidate xb values for Bmin scaling.
        :type xb_grid: list[float] or None
        :param xh_grid: Candidate xh values for Hmax scaling.
        :type xh_grid: list[float] or None
        :param z_grid: Candidate integer values for Zmax.
        :type z_grid: list[int] or None
        :param lam_c_grid: Candidate SD cost-penalty weights.
        :type lam_c_grid: list[float] or None
        :param lam_t_grid: Candidate SD time-penalty weights.
        :type lam_t_grid: list[float] or None
        :param weight_mode: Weight family used in ranking ("inv" or "exp").
        :type weight_mode: str
        :param trials: Rollouts per parameter configuration.
        :type trials: int
        :param horizon: Simulation horizon per rollout.
        :type horizon: int
        :param warmup: Warmup periods excluded from averaging.
        :type warmup: int
        :param base_seed: Base seed used to generate rollout seeds.
        :type base_seed: int

        :return: Tuple (best_params, best_result) where best_result is a held-out eval.
        :rtype: tuple[AMBSParamsSD, AMBSResult]
        """
        K = self.inst.K
        if xb_grid is None:
            xb_grid = [i / 10.0 for i in range(0, 11)]
        if xh_grid is None:
            xh_grid = [i / 10.0 for i in range(5, 11)]
        if z_grid is None:
            z_grid = list(range(1, max(2, K)))

        if lam_c_grid is None:
            lam_c_grid = [0.0, 0.5, 1.0]
        if lam_t_grid is None:
            lam_t_grid = [0.0, 0.5, 1.0]

        combos = [(xb, xh, z, lc, lt)
                  for xb in xb_grid
                  for xh in xh_grid
                  for z  in z_grid
                  for lc in lam_c_grid
                  for lt in lam_t_grid]

        seeds = [int(base_seed + r) for r in range(int(trials))]

        best_par = None
        best_cost = None

        with tqdm(total=len(combos) * len(seeds), desc="Grid AMBS-SD", leave=False) as pbar:
            for xb, xh, z, lc, lt in combos:
                Bmin, Hmax, Zmax = make_thresholds(self.inst, xb, xh, z)
                par = AMBSParamsSD(Bmin, Hmax, Zmax, lam_c=lc, lam_t=lt, weight_mode=weight_mode)

                costs = []
                for s in seeds:
                    res = self.simulate(par, horizon=horizon, warmup=warmup, seed_rollout=s, show_pbar=False)
                    costs.append(res.avg_cost)
                    pbar.update(1)

                avg_cost = float(np.mean(costs))
                if (best_cost is None) or (avg_cost < best_cost):
                    best_cost = avg_cost
                    best_par = par

        final_eval = self.simulate(
            best_par,
            horizon=horizon,
            warmup=warmup,
            seed_rollout=int(base_seed + 999),
            show_pbar=False
        )
        return best_par, final_eval

# ============================================================
# 8) Execution: (i) threshold+lambda calibration; (ii) evaluation
# ============================================================
inst, exp_name = build_inst_from_instance_id(INSTANCE_ID)

ambs_sd = AMBSSDHeuristicOnline(inst, seed=SEED)

# ---- Calibration ----
best_params, best_result = ambs_sd.grid_search(
    xb_grid=[i / 10.0 for i in range(0, 11)],
    xh_grid=[i / 10.0 for i in range(5, 11)],
    z_grid=list(range(1, inst.K)),
    lam_c_grid=[0.0, 0.5, 1.0],
    lam_t_grid=[0.0, 0.5, 1.0],
    weight_mode="inv",
    trials=10,
    horizon=1000,
    warmup=100,
    base_seed=12345,
)

print("\n==============================================")
print(f"AMBS-SD (online) experiment: {exp_name}")
print("== Best parameters found (grid) ==")
print(f"Bmin = {best_params.Bmin:.4f}, Hmax = {best_params.Hmax:.4f}, Zmax = {best_params.Zmax}")
print(f"lam_c = {best_params.lam_c:.3f}, lam_t = {best_params.lam_t:.3f}, weight_mode={best_params.weight_mode}")
print(f"Cost (grid evaluation) = {best_result.avg_cost:.3f}")
print("==============================================\n")

# ---- FINAL evaluation ----
N_RUNS = 10
HORIZON = 10_000
WARMUP = 1_000
SEED_BASE_TEST = 12345

run_costs, run_inv, run_bo, run_setup = [], [], [], []
run_setups_rate, run_lot_units = [], []

print("== Individual AMBS-SD results (same seeds as DeepRL test) ==")
for r in range(N_RUNS):
    seed_r = SEED_BASE_TEST + r
    res_r = ambs_sd.simulate(best_params, horizon=HORIZON, warmup=WARMUP, seed_rollout=seed_r, show_pbar=False)

    print(f"\n-- Run {r + 1}/{N_RUNS} (seed {seed_r}) --")
    print(f"Average total cost : {res_r.avg_cost:.3f}")
    print(f" - Inventory : {res_r.avg_inv_cost:.3f}")
    print(f" - Backorder : {res_r.avg_bo_cost:.3f}")
    print(f" - Setups : {res_r.avg_setup_cost:.3f}")
    print(f"Setups / period : {res_r.avg_setups:.3f}")
    print(f"Avg units / period : {res_r.avg_lot_size_units:.3f}")
    print(f"Step 3 triggers : {res_r.step3_count}")

    run_costs.append(res_r.avg_cost)
    run_inv.append(res_r.avg_inv_cost)
    run_bo.append(res_r.avg_bo_cost)
    run_setup.append(res_r.avg_setup_cost)
    run_setups_rate.append(res_r.avg_setups)
    run_lot_units.append(res_r.avg_lot_size_units)

print("\n== Average over 10 runs (horizon = 10k, warmup = 1k) ==")
print(f"Average total cost : {float(np.mean(run_costs)):.3f}")
print(f" - Inventory : {float(np.mean(run_inv)):.3f}")
print(f" - Backorder : {float(np.mean(run_bo)):.3f}")
print(f" - Setups : {float(np.mean(run_setup)):.3f}")
print(f"Setups / period : {float(np.mean(run_setups_rate)):.3f}")
print(f"Avg units / period : {float(np.mean(run_lot_units)):.3f}")
print("==============================================")